In [0]:
# =============================================================================
# TRAVEL BOOKING SCD2 MERGE PROJECT - BRONZE LAYER: BOOKING DATA INGESTION
# =============================================================================

from pyspark.sql.functions import current_timestamp
from pyspark.sql import functions as F

# =============================================================================
# PARAMETER EXTRACTION WITH DEFAULTS
# =============================================================================
# same as validate input

import datetime as _dt
try:
    arrival_date = dbutils.widgets.get("arrival_date")
except Exception:
    arrival_date = _dt.date.today().strftime("%Y-%m-%d")
try:
    catalog = dbutils.widgets.get("catalog")
except Exception:
    catalog = "travel_bookings"
try:
    schema = dbutils.widgets.get("schema")
except Exception:
    schema = "default"
try:
    base_volume = dbutils.widgets.get("base_volume")
except Exception:
    base_volume = f"/Volumes/{catalog}/{schema}/data"

# =============================================================================
booking_path = f"{base_volume}/booking_data/bookings_{arrival_date}.csv"

df=spark.read.option("header","true").option("inferSchema","true").option("quote","\"").option("multiLine","true").csv(booking_path)

df = df.withColumn("ingestion_time", current_timestamp()) \
       .withColumn("business_date", F.to_date(F.lit(arrival_date)))

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.bronze")

df.write.format("delta").mode("append").saveAsTable(f"{catalog}.bronze.booking_inc")

print(f"Ingested rows: {df.count()}")